# Load a ZEN reference spectrum

This example reads the single signature in the bundled ZEN linear-unmixing
export, `data/Argo_405.xml`. A Lumic `Spectrum` keeps the exported intensity
values, wavelength labels, name, and selected acquisition information together.

Install Lumic from this checkout before running the notebook. From the
repository root, with the notebook's Python 3.12 environment active, run
`python -m pip install -e './Python/lumic[examples]'`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from lxml import etree

from lumic import Spectrum

# Find the bundled fixture from the notebook folder or repository root.
cwd = Path.cwd().resolve()
for folder in (cwd / "Python", cwd, *cwd.parents):
    xml_path = folder / "data" / "Argo_405.xml"
    if xml_path.is_file():
        break
else:
    raise FileNotFoundError("Open the notebook from inside the Lumic checkout to find Argo_405.xml.")

## Read the exported values

This ZEN export stores one wavelength–intensity pair per row in
`SpectrumData`. Rows are separated by semicolons. The wavelengths and
intensities go into the object in file order and on their original scale.

In [ ]:
parser = etree.XMLParser(resolve_entities=False)
root = etree.parse(str(xml_path), parser).getroot()
if root.tag != "LinearUnmixingDyeSpectrum":
    raise ValueError("Expected a ZEN LinearUnmixingDyeSpectrum export.")

spectrum_text = root.findtext("SpectrumData") or ""
rows = [row for row in spectrum_text.split(";") if row.strip()]
if not rows:
    raise ValueError("The ZEN export has no samples in SpectrumData.")

samples = np.loadtxt(rows, ndmin=2)
if samples.shape[1] != 2:
    raise ValueError("Expected a wavelength and intensity on each SpectrumData row.")

spectrum = Spectrum(
    values=samples[:, 1],
    wavelengths_nm=samples[:, 0],
    name=root.get("Name"),
    provenance={
        "format": "ZEN XML",
        "path": str(xml_path),
        "acquisition_date": root.findtext("AcquisitionDate"),
        "objective": root.findtext("Objective"),
    },
)

print(f"{spectrum.name}: {spectrum.values.size} samples")
print(spectrum.provenance)

## Plot the spectrum

`spectrum.values` is the one-dimensional NumPy array. Before using this
signature for unmixing, check that its detector channels and wavelength
labels correspond to the channels of the target image.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.plot(spectrum.wavelengths_nm, spectrum.values, marker=".")
ax.set(title=spectrum.name, xlabel="Wavelength (nm)", ylabel="Exported intensity (a.u.)")
plt.show()